In [18]:
import pathlib
import os
import datetime

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import torch
from torch.utils.tensorboard.writer import SummaryWriter
import trimesh

import utils, dataset, train
from encoder import *
from decoder import *

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [41]:
# Model name
current_time = datetime.datetime.now().strftime("%b%d_%H-%M")
denoiser_name = "3_Objects_100epochs_64batch_KL0.0001"
experiment_name = f"{current_time}_{denoiser_name}"


config = {
    'experiment_name': experiment_name,
    'device': 'cuda:0',
    'batch_size': 64,
    'resume_ckpt': None,
    'learning_rate': 0.0004,
    'step_size': 10, # scheduler step, one step is one batch
    'gamma': 1,
    'max_epochs': 100,
    'timesteps': 1000,
    'print_every_n': 15, # every n batches
    # 'validate_every_n': 10,
    # 'print_EMD_every_n': 1
}

model_config = {
    'last_epoch': 0,
}

In [42]:
# declare device
if torch.cuda.is_available() and config['device'].startswith('cuda'):
    device = torch.device(config['device'])
    print('Using device:', config['device'])
else:
    device = torch.device('cpu')
    print('Using CPU')

# create dataloaders
trainset = dataset.Dataset('overfit', config['timesteps'])
trainloader = torch.utils.data.DataLoader(trainset, batch_size=config['batch_size'], shuffle=True, num_workers=0, pin_memory=False)
# valset = dataset.Dataset('val', config['timesteps'])
# valloader = torch.utils.data.DataLoader(valset, batch_size=config['batch_size'], shuffle=False, num_workers=0)

encoder = Encoder()
decoder = Decoder()

# move model to specified device
encoder.to(device)
decoder.to(device)
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=config['learning_rate'])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, config['step_size'], config['gamma'])
# scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.0006, steps_per_epoch=len(trainloader), epochs=config['max_epochs'], anneal_strategy='cos', three_phase=True)



# load model if resuming from checkpoint
# experiment_name = "May31_17-04_10_Objects_200epochs_32batch"
# utils.reload_model(encoder, decoder, optimizer, None, experiment_name, 'epoch136', device)


total_enc, trainable_enc = utils.count_parameters(encoder)
total_dec, trainable_dec = utils.count_parameters(decoder)
print(f"Encoder Total: {total_enc:,} | Trainable: {trainable_enc:,} | Model size: {utils.model_memory_size(encoder):.3f} MB")
print(f"Decoder Total: {total_dec:,} | Trainable: {trainable_dec:,} | Model size: {utils.model_memory_size(decoder):.3f} MB")

Using device: cuda:0
Encoder Total: 159,360 | Trainable: 159,360 | Model size: 0.608 MB
Decoder Total: 188,632 | Trainable: 188,632 | Model size: 0.739 MB


In [40]:
torch.cuda.empty_cache()

In [43]:
# start training
torch.cuda.empty_cache()
#tensorboard --logdir=diffusion_autoencoder/logs
# config['max_epochs'] = 200
# Create tensorboard writer    
log_path = pathlib.Path(f"logs/{datetime.datetime.now().strftime('%b%d')}/{config['experiment_name']}")
writer = SummaryWriter(log_path)

with open("debug.txt", "w", encoding="utf-8") as debug_file:
    train.train(encoder, decoder, trainloader, None, device, optimizer, scheduler, config, model_config, writer, None, debug_file)

writer.close()


[00/014] train_loss: 1.527
Best Model:   1.5155471190810204
[01/013] train_loss: 1.372
Best Model:   1.3695360198616982
[02/012] train_loss: 1.286
Best Model:   1.26640984416008
[03/011] train_loss: 1.128
Best Model:   1.0682295635342598
[04/010] train_loss: 0.845
Best Model:   0.7746732383966446
[05/009] train_loss: 0.642
Best Model:   0.5755551401525736
[06/008] train_loss: 0.465
Best Model:   0.40005459636449814
[07/007] train_loss: 0.332
Best Model:   0.27756943088024855
[08/006] train_loss: 0.248
Best Model:   0.21690642833709717
[09/005] train_loss: 0.190
Best Model:   0.17028509080410004
[10/004] train_loss: 0.165
[11/003] train_loss: 0.185
Best Model:   0.1558313276618719
[12/002] train_loss: 0.164
[13/001] train_loss: 0.168
[14/000] train_loss: 0.165
[14/015] train_loss: 0.177
[15/014] train_loss: 0.186
[16/013] train_loss: 0.178
[17/012] train_loss: 0.168
[18/011] train_loss: 0.154
[19/010] train_loss: 0.166
Best Model:   0.15538522182032466
[20/009] train_loss: 0.156
Best Mo

In [ ]:
#Load a model:
experiment_name = ""
utils.reload_model(encoder, decoder, None, None, experiment_name, 'checkpoint', device)

({'experiment_name': 'May20_23-27_Resnet_2_256_80epochs_32batch',
  'device': 'cuda:0',
  'is_overfit': True,
  'batch_size': 32,
  'resume_ckpt': None,
  'learning_rate': 0.0004,
  'step_size': 10,
  'gamma': 1,
  'max_epochs': 200,
  'timesteps': 1000,
  'print_every_n': 2},
 {'denoiser_class': 'Denoiser', 'last_epoch': 171})

In [46]:
generateDDIM = 1
generateDDPM = 0
number_of_points = 2048
DDIM_steps = 50
number_of_DDIM_iterations = 2
save = False
start = 20

if generateDDIM:
    for i in range(start, start + number_of_DDIM_iterations):
        object_index = 2
        mean, log_variance = encoder(trainset[object_index].unsqueeze(0))
        code = encoder.sample_latent_z(mean, log_variance)
        generated_pc_ddim = sample_ddim(decoder, code, n_points=number_of_points, steps=DDIM_steps)
        generated_pcd_ddim = trimesh.PointCloud(generated_pc_ddim.squeeze().cpu().numpy())
        utils.visualize_comparison(trainset[object_index], generated_pc_ddim, window_name="DDIM Target (Red) vs Generated (Blue)")
        if save:
            path = pathlib.Path(f"output/{experiment_name}")
            path.mkdir(parents=True, exist_ok=True)
            generated_pcd_ddim.export(path / f"ddim{DDIM_steps}_{number_of_points}_{i}.obj")

# if generateDDPM:
#     generated_pc_ddpm = network.sample_ddpm(denoiser, diffuser, n_points=number_of_points)
#     generated_pcd_ddpm = trimesh.PointCloud(generated_pc_ddpm.squeeze().cpu().numpy())
#     utils.visualize_comparison(trainset[0], generated_pc_ddpm, window_name="DDPM Target (Red) vs Generated (Blue)")

Visualizing: Target is RED, Generated is BLUE.
Visualizing: Target is RED, Generated is BLUE.


In [ ]:
#VAE latent space visualization

In [ ]:
#Decoder interpolation

In [14]:
# Optionally, save the generated point clouds to disk
path = pathlib.Path(f"output/{experiment_name}")
path.mkdir(parents=True, exist_ok=True)
generated_pcd_ddim.export(path / f"ddim{DDIM_steps}_{number_of_points}.obj")
# generated_pcd_ddpm.export(f"output/{experiment_name}_ddpm.obj")